In [0]:
print("Hello there")

In [0]:
from pathlib import Path
import pandas as pd

# ####### PATH FOR DATA FILES IN VSCode ######
# #Create a data folder for the files generated by this notebook.
# from pathlib import Path
# DATA_DIR = Path("../data") #./data Actual level ---- ../data 2 levels up
# imdb_path = DATA_DIR / "IMDB TMDB Movie Metadata Big Dataset (1M) ORIGINAL.csv"
# links_path = DATA_DIR / "links ORIGINAL.csv"

# # 0. Loading datasets imbd and links 
# # Those two datasets are the conection between each movie and its information
# imdb_df = pd.read_csv(imdb_path)
# links_df = pd.read_csv(links_path)


# ####### PATH FOR DATA FILES IN DATABRICKS######
# These are Unity Catalog tables, not CSV files
# Read from Unity Catalog tables and convert to pandas
imdb_df = spark.read.table("workspace.datasets.imdb_big_dataset").toPandas()
links_df = spark.read.table("workspace.datasets.links").toPandas()
# display(imdb_df)
# display(links_df)


#### ----- Joint databases ----------

# 1. The Id at IMDB is not just numbers, then the first step is to 
# standardize ID de IMDB (delete 'tt' and transform to int)

imdb_df['imdb_id_clean'] = imdb_df['imdb_id'].str.replace('tt', '').astype(float).fillna(0).astype(int)
#imdb_df.head(10)
#links_df.head(10)

# 3. Join and merge the datasets 
# ("how = inner" merge ensure consistency)
# 'imdbId' is the conection between both datasets links.csv 
movies_combined = pd.merge(imdb_df, links_df, left_on='imdb_id_clean', right_on='imdbId', how='inner')
# how='inner' hace un inner join:
# Keep just rows that has coincidence in both dataframes
# This remove the rows that dont have equivalent key in both dataframes

# # 4. Select necessary rows to design the MVP:
columns_to_keep = [
    'title', 'genres_list', 'overview', 'keywords', 'Cast_list', 
    'Director', 'vote_average', 'release_date', 'original_language', 'movieId'
]
movies_final = movies_combined[columns_to_keep].copy()

In [0]:
imdb_df.head(3)
links_df.head(3)
movies_combined.head(3)
movies_final.head(5)

# Cleaning  

In [0]:
###################    NULL MANAGMENT ###########################################

# Check null cells
# movies_final['overview'].isnull().sum() # .isnull or .isna() # Per column
# print(movies_final.isnull().sum())

# If the null case happens clean it with white spaces
# movies_final ['overview'] = movies_final ['overview'].fillna('')



################### GENRE NORMALIZATION ##########################################
# Check spaces and punctuaction 
# # Check if any genres have leading/trailing spaces
movies_final['genres_list'].str.contains('^\\s|\\s$', regex=True).sum()

###----- Function to remove spaces from each item in the list 
movies_final['genres_list'] = movies_final['genres_list'].apply(
    lambda x: [genre.strip() for genre in x] if isinstance(x, list) else x
)

### ----- Check for punctuation
# movies_final['genres_list'].str.contains(f'[{string.punctuation}]', regex=True).sum()

### Remove punctuation from genre strings ----- If it is necesary after cheking 
# import string
# movies_final['genres_list'] = movies_final['genres_list'].apply(
#     lambda x: [genre.translate(str.maketrans('', '', string.punctuation)) for genre in x] 
#     if isinstance(x, list) else x
# )


### ----- Verify all genres are clean (no spaces at edges, no punctuation) ---- If is required to cleaning
# def is_clean(genres_list):
#     if not isinstance(genres_list, list):
#         return False
#     return all(
#         genre == genre.strip() and  # No leading/trailing spaces
#         not any(c in genre for c in string.punctuation)  # No punctuation
#         for genre in genres_list
#     )
# movies_final['genres_list'].apply(is_clean).sum()  # Count clean rows

######### SAVING DATAFRAME AS CSV IN VSCode #############################
# #### ----- Saving the Dataframe as CSV ---
# movies_final.to_csv("../data/clean_movies.csv", index=False)

######### SAVING DATAFRAME AS CSV IN DATABRICKS #############################
# #### ----- Saving the Dataframe as CSV ---
# Convertir a Spark DataFrame 
spark_df = spark.createDataFrame(movies_final) 
# Guardar como tabla en el Catalog 
spark_df.write.mode("overwrite").saveAsTable("datasets.clean_movies")


In [0]:
movies_final.head(5)

Overview and keywords can not be null, those has to be filled with zero if that happens